In [2]:
import sys; sys.path.append("../src")
import pandas as pd
from engine import *
from research_stats import *

df = load_clean()
END = "2017-12-31"
rows = []

for thr in [-0.015, -0.02, -0.03]:
    for h in [1, 3, 5, 10]:
        cfg = Config(threshold=thr, hold=h)
        ev   = get_events(df, cfg, end=END)
        base = get_baseline(df, cfg, end=END)
        r = diff_test(ev["fwd_ret"], base["fwd_ret"], block=20, n_boot=2000)
        rows.append({
            "thr_%": thr * 100,
            "hold": h,
            "n_events": len(ev),
            "ev_mean_%": ev["fwd_ret"].mean() * 100,
            "base_mean_%": base["fwd_ret"].mean() * 100,
            "diff_%": r["diff_%"],
            "ci_lo_%": r["ci_lo_%"],
            "ci_hi_%": r["ci_hi_%"],
            "p_one_sided": r["p_one_sided"],
            "net_ev_mean_%": (ev["fwd_ret"].mean() - 2 * cfg.cost_per_side) * 100,
        })

grid = pd.DataFrame(rows).round(3)
print(grid.to_string(index=False))
print("Bonferroni p-value cutoff for", len(grid), "tests:", round(0.05 / len(grid), 4))

 thr_%  hold  n_events  ev_mean_%  base_mean_%  diff_%  ci_lo_%  ci_hi_%  p_one_sided  net_ev_mean_%
  -1.5     1       245     -0.105       -0.035  -0.071   -0.351    0.214        0.691         -0.205
  -1.5     3       177      0.116        0.037   0.079   -0.455    0.612        0.388          0.016
  -1.5     5       146      0.325        0.103   0.223   -0.428    0.914        0.246          0.225
  -1.5    10       103      0.202        0.312  -0.110   -1.257    1.018        0.581          0.102
  -2.0     1       145     -0.178       -0.033  -0.144   -0.557    0.265        0.752         -0.278
  -2.0     3       111      0.192        0.041   0.151   -0.581    0.875        0.348          0.092
  -2.0     5        92      0.070        0.118  -0.049   -0.984    0.934        0.539         -0.030
  -2.0    10        70      0.709        0.315   0.394   -1.029    1.964        0.299          0.609
  -3.0     1        60     -0.245       -0.037  -0.209   -1.096    0.638        0.685      

Robustness grid: 3 thresholds x 4 holding periods = 12 tests, development data only.
The primary test was fixed in advance (-2%, hold 5). The other 11 cells are only a
stability check. I will NOT pick the best cell. With 12 tests, one "significant" result
is expected by chance, so I compare p-values against the Bonferroni cutoff, not 0.05.